# Visualize Persona Vectors in Assistant Axis Landscape

This notebook uses modular utilities to:
- Load pre-computed role vectors and assistant axis from HuggingFace
- Fit PCA to create the landscape
- Project custom persona vectors (e.g., from fine-tuning)
- Visualize in 2D and 3D interactive plots
- Compute quantitative insights (nearest neighbors, alignment)

**Supported models:**
- `gemma-2-27b` (layer 22)
- `llama-3.3-70b` (layer 40)
- `qwen-3-32b` (layer 32)

In [12]:
from pathlib import Path
import torch
from huggingface_hub.utils import disable_progress_bars

import landscape_utils as lu

# Suppress HF progress bars
disable_progress_bars()

## Configuration

In [13]:
# Model configuration
MODEL_NAME = "gemma-2-27b"  # Options: "gemma-2-27b", "llama-3.3-70b", "qwen-3-32b"
TARGET_LAYER = 22  # Recommended: gemma=22, llama=40, qwen=32

# HuggingFace dataset
REPO_ID = "lu-christina/assistant-axis-vectors"

print(f"Model: {MODEL_NAME}")
print(f"Target layer: {TARGET_LAYER}")

Model: gemma-2-27b
Target layer: 22


## Load Vectors and Fit PCA

In [14]:
# Load pre-computed vectors from HuggingFace
role_vectors, default_vector, assistant_axis, local_dir = lu.load_vectors_from_hf(
    MODEL_NAME, REPO_ID
)

✓ Loaded 275 role vectors
  Default vector shape: torch.Size([46, 4608])
  Assistant axis shape: torch.Size([46, 4608])


In [15]:
# Fit PCA on role vectors
pca, scaler, pca_result, role_labels, variance_explained, role_vectors_centered = lu.fit_pca_landscape(
    role_vectors, TARGET_LAYER
)

Fitting PCA on 275 vectors at layer 22...
  Vector shape: (275, 4608)
✓ PCA fitted with 275 components
  First 3 PCs: 48.8%, 9.8%, 7.0%
  Cumulative (first 3): 65.6%


In [16]:
# Project default assistant and axis into PC space
default_pc = lu.project_vector(default_vector[TARGET_LAYER], scaler, pca, is_difference_vector=False)
assistant_axis_pc = lu.project_vector(assistant_axis[TARGET_LAYER], scaler, pca, is_difference_vector=True)

# Get original axis for coloring
assistant_axis_orig = assistant_axis[TARGET_LAYER].float().numpy()

print(f"Default assistant PC (first 3): {default_pc[:3]}")
print(f"Assistant axis PC (first 3): {assistant_axis_pc[:3]}")

Default assistant PC (first 3): [674.28394  -21.292511  65.77762 ]
Assistant axis PC (first 3): [659.74725  -25.354748  65.80742 ]


## Load Custom Vectors

Load persona vectors from multiple directories (e.g., base model, fine-tuned models, etc.)

In [ ]:
# Paths to custom vector directories with optional aliases
# Format: (path, alias) or just path
# If alias is provided, it will be prepended to the vector name
CUSTOM_VECTOR_PATHS = [
    ("../extract-ft-persona-vector/outputs-base-model/vectors", "Base (8 samples)"),
    ("../extract-ft-persona-vector/outputs-base-model-full/vectors", "Base"),
    ("../extract-ft-persona-vector/outputs/vectors", "LoRA (8 samples)"),
    ("../extract-ft-persona-vector/outputs-full/vectors", "LoRA"),
    # Add more: ("path/to/vectors", "MyAlias") or just "path/to/vectors"
]

# Optionally override specific vector names entirely
# Format: {filename: display_name}
CUSTOM_NAMES = {
    # Example: "gemma-2-27b-it-base.pt": "Base Model (8 samples)",
    # Example: "avoid-letter-o.pt": "Avoid-O Fine-tuned",
}

# Load all .pt files from all directories
custom_vectors = {}
loaded_files = []

for path_spec in CUSTOM_VECTOR_PATHS:
    # Handle both (path, alias) tuples and plain path strings
    if isinstance(path_spec, tuple):
        vector_dir, alias = path_spec
    else:
        vector_dir = path_spec
        alias = None
    
    vector_path = Path(vector_dir)
    
    if not vector_path.exists():
        print(f"⚠ Directory not found: {vector_path}")
        continue
    
    print(f"\nLoading from: {vector_path}" + (f" (alias: {alias})" if alias else ""))
    
    for pt_file in sorted(vector_path.glob("*.pt")):
        # Load and project
        vector_at_layer, metadata = lu.load_custom_vector(str(pt_file), TARGET_LAYER)
        
        # Determine display name
        if pt_file.name in CUSTOM_NAMES:
            # Full override from CUSTOM_NAMES
            vector_name = CUSTOM_NAMES[pt_file.name]
        elif alias:
            # Use alias prefix
            vector_name = f"{alias}: {metadata['checkpoint']}"
        else:
            # Use checkpoint name as-is
            vector_name = metadata['checkpoint']
        
        # Handle duplicates by adding directory name
        if vector_name in custom_vectors:
            vector_name = f"{vector_name} ({vector_path.name})"
        
        # Project to PC space
        vector_pc = lu.project_vector(vector_at_layer, scaler, pca, is_difference_vector=False)
        custom_vectors[vector_name] = vector_pc
        loaded_files.append(str(pt_file))
        
        print(f"  ✓ {pt_file.name} -> '{vector_name}'")
        print(f"    PC (first 3): {vector_pc[:3]}")

print(f"\n{'='*60}")
print(f"✓ Loaded {len(custom_vectors)} custom vectors from {len(CUSTOM_VECTOR_PATHS)} directories")
print(f"{'='*60}")

## Analyze Custom Vectors

In [18]:
# Analyze each custom vector
analyses = {}

for name, vector_pc in custom_vectors.items():
    analysis = lu.analyze_vector(
        vector_name=name,
        vector_pc=vector_pc,
        default_pc=default_pc,
        assistant_axis_pc=assistant_axis_pc,
        pca_result=pca_result,
        role_labels=role_labels,
        n_dims=3,
        top_k=5,
        verbose=True
    )
    analyses[name] = analysis
    print()


Analysis for: gemma-2-27b-it-base
  PC coordinates (first 3): [1269.1393    112.70674    80.671875]

Alignment metrics:
  Distance to default: 609.94
  Cosine to axis: 0.9914
  Projection on axis: 1265.64
  Perpendicular distance: 167.54

Nearest role vectors (top 5):
  debugger: 318.91
  statistician: 327.44
  reviewer: 340.27
  validator: 350.09
  scientist: 379.40


Analysis for: gemma-2-27b-it-base (vectors)
  PC coordinates (first 3): [1418.2205   -111.150116  -80.33709 ]

Alignment metrics:
  Distance to default: 763.46
  Cosine to axis: 0.9871
  Projection on axis: 1406.47
  Perpendicular distance: 228.06

Nearest role vectors (top 5):
  examiner: 374.36
  grader: 430.48
  debugger: 431.60
  screener: 450.54
  editor: 492.95


Analysis for: gemma-test-20260211-221245
  PC coordinates (first 3): [ 943.73645 -321.77408 -183.34633]

Alignment metrics:
  Distance to default: 474.30
  Cosine to axis: 0.9198
  Projection on axis: 932.50
  Perpendicular distance: 397.79

Nearest role 

## 2D Visualization

In [19]:
fig_2d = lu.plot_2d_landscape(
    pca_result=pca_result,
    role_labels=role_labels,
    variance_explained=variance_explained,
    default_pc=default_pc,
    assistant_axis_pc=assistant_axis_pc,
    role_vectors_centered=role_vectors_centered,
    assistant_axis_orig=assistant_axis_orig,
    test_vectors=custom_vectors,
    title=f"{MODEL_NAME} - Persona Vectors in Assistant Axis Landscape (2D)",
    show_axis_line=True
)

fig_2d.show()

## 3D Visualization

In [20]:
fig_3d = lu.plot_3d_landscape(
    pca_result=pca_result,
    role_labels=role_labels,
    variance_explained=variance_explained,
    default_pc=default_pc,
    assistant_axis_pc=assistant_axis_pc,
    role_vectors_centered=role_vectors_centered,
    assistant_axis_orig=assistant_axis_orig,
    test_vectors=custom_vectors,
    title=f"{MODEL_NAME} - Persona Vectors in Assistant Axis Landscape (3D)",
    show_axis_line=True
)

fig_3d.show()

In [22]:
# Select vectors to compare (modify as needed)
vectors_to_compare = {
    name: pc for name, pc in custom_vectors.items()
    # Add filters here if needed
}

print(f"Comparing {len(vectors_to_compare)} vectors")

# 2D comparison
fig_compare_2d = lu.plot_2d_landscape(
    pca_result=pca_result,
    role_labels=role_labels,
    variance_explained=variance_explained,
    default_pc=default_pc,
    assistant_axis_pc=assistant_axis_pc,
    role_vectors_centered=role_vectors_centered,
    assistant_axis_orig=assistant_axis_orig,
    test_vectors=vectors_to_compare,
    title=f"{MODEL_NAME} - Vector Comparison (2D)",
    show_axis_line=True
)

fig_compare_2d.show()

Comparing 3 vectors
